In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
load_dotenv()  # Load environment variables from .env file

In [ ]:
WORKING_MODELS = {
    "qwen"    : {"model": "Qwen/Qwen2.5-72B-Instruct",          "provider": "novita"},
    "llama"   : {"model": "meta-llama/Llama-3.1-8B-Instruct",   "provider": "novita"},
    "deepseek": {"model": "deepseek-ai/DeepSeek-R1",             "provider": "novita"},

    "gemma"   : {"model": "google/gemma-2-9b-it",                "provider": "together"},
    "gemma_sm": {"model": "google/gemma-2-2b-it",                "provider": "together"},
}

In [ ]:
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
from openai import OpenAI
import os

load_dotenv()

PROVIDER = os.getenv("PROVIDER", "huggingface")

if PROVIDER == "openai":
    client = OpenAI()

    def call_llm(prompt):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content.strip()

elif PROVIDER == "huggingface":
    client = InferenceClient(
        provider="novita",
    )

    def call_llm(prompt):
        response = client.chat.completions.create(
            model=os.getenv("HF_MODEL", "deepseek-ai/DeepSeek-R1"),
            messages=[{"role": "user", "content": prompt}],
           
        )
        return response.choices[0].message.content.strip()


# test
print(call_llm("what is 2+2? one word answer"))

In [ ]:
email_1 = {
    "subject": "Can't login — paid for annual plan last week",
    "body"   : "Our entire team can't login. We paid for annual "
               "plan last week. Board demo in 3 hours.",
    "correct": "Billing"  
}

subject = email_1['subject']
body    = email_1['body']


In [ ]:

PROMPT_WITH_CONFIDENCE = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

Also give a Confidence Score from 1-10:
- 10 = absolutely sure
- 5  = could go either way
- 1  = total guess

Rules:
- Return ONLY in this exact format:  Category | Score
- Example outputs:  Billing | 9
- Example outputs:  Technical | 6
- Example outputs:  Other | 3
- No explanation. Nothing else.

EMAIL TO CLASSIFY:
Subject : {subject}
Body    : {body}"""


raw        = call_llm(PROMPT_WITH_CONFIDENCE)


In [ ]:
raw

# A new support agent joins your team on Monday.

# A ticket comes in:

#   Subject: "Can't login — paid for annual plan last week"
#   Body: "Our entire team can't login. We paid for 
#          annual plan last week. Board demo in 3 hours."

# New agent thinks:
# "Can't login → Technical team" 
# Assigns to: Engineering

# Senior agent thinks:
# "Wait... they paid last week...
#  Annual plan payments need manual activation...
#  Account was never activated after payment...
#  Login is broken BECAUSE of billing...
#  This is a Billing issue!" 
# Assigns to: Billing team

In [ ]:
PROMPT_CRAFTED = f"""You are a senior support agent with 10 years 
of experience at a B2B SaaS company.

You have seen hundreds of tickets. You know that:
- "Can't login" is often a SYMPTOM, not the root cause
- Recent payment + login issue = account not activated = Billing
- Always ask yourself WHY before classifying

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Rules:
- Login broken because of recent payment → Billing, NOT Technical
- Sarcastic complaints ("wow great job") → still Billing
- Auto-reply / out-of-office → Other
- Always classify by ROOT CAUSE, not surface words

Subject : {subject}
Body    : {body}

Return ONLY: Category | Confidence (1-10)"""

result = call_llm(PROMPT_CRAFTED)
result

In [ ]:
emails = [
    {
        "subject": "Can't login — paid for annual plan last week",
        "body"   : "Our entire team can't login. We paid for annual "
                   "plan last week. Board demo in 3 hours.",
        "correct": "Billing"
    },
    {
        "subject": "Quick question about API rate limits",
        "body"   : "When do rate limits reset? We're processing "
                   "payroll for 5000 employees tonight.",
        "correct": "Technical + High Urgency"
    },
    {
        "subject": "Evaluating if we stay on your platform",
        "body"   : "Zapier missing, bulk export missing, "
                   "pricing jumped 40%.",
        "correct": "Billing"
    }
]

In [ ]:
email = {
    "subject": "NACH mandate registration failed",
    "body"   : "Our NACH mandate for auto-debit failed. "
               "Customer accounts cannot be debited for EMI this month.",
    "correct": "Technical"
}

subject = email['subject']
body    = email['body']

PROMPT_ZERO_SHOT = f"""You are an expert support email classifier.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Return ONLY: Category | Confidence (1-10)"""

PROMPT_FEW_SHOT = f"""You are an expert support email classifier.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

EXAMPLES:
Subject: SFDC integration not syncing
Body: Salesforce stopped pulling leads.
Output: Technical | 9

Subject: NACH mandate failed
Body: Auto-debit mandate registration failed for EMI collection.
Output: Technical | 10

Subject: UPI reversal not credited
Body: Payment failed but amount not refunded yet.
Output: Billing | 9

NOW CLASSIFY:
Subject : {subject}
Body    : {body}

Return ONLY: Category | Confidence (1-10)"""

zero_shot = call_llm(PROMPT_ZERO_SHOT)
few_shot  = call_llm(PROMPT_FEW_SHOT)

print(f"Email     : {email['subject']}")
print(f"Expected  : {email['correct']}")
print(f"Zero-Shot : {zero_shot}")
print(f"Few-Shot  : {few_shot}")

In [ ]:
# Zero-Shot → no examples, model guesses
# Few-Shot  → examples diye, model learns pattern
# Role-Based → persona dete hain, model acts like expert



# Same question: "Is this email urgent?"

# Intern:
# "Umm... it says quick question... so Low?" ❌

# Senior Customer Success Manager with 10 years experience:
# "Payroll for 5000 employees tonight + rate limit...
#  If this isn't fixed in 1 hour — 
#  5000 people won't get salary.
#  This is CRITICAL." ✅

In [ ]:
email = {
    "subject": "Quick question about API rate limits",
    "body"   : "When do rate limits reset? We're processing "
               "payroll for 5000 employees tonight.",
    "correct": "Technical | High"
}

subject = email['subject']
body    = email['body']

# Without Role
PROMPT_NO_ROLE = f"""Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Also classify urgency: High | Medium | Low

Subject : {subject}
Body    : {body}

Return ONLY: Category | Urgency | Confidence (1-10)"""

# With Role
PROMPT_WITH_ROLE = f"""You are a Senior Customer Success Manager
with 10 years experience at a B2B SaaS company.

You understand:
- "Quick question" does NOT mean low urgency
- Always read the BUSINESS IMPACT, not just the words
- Payroll, demos, client meetings = always High urgency
- 5000+ employees affected = Critical/High always

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Also classify urgency: High | Medium | Low

Subject : {subject}
Body    : {body}

Return ONLY: Category | Urgency | Confidence (1-10)"""

no_role   = call_llm(PROMPT_NO_ROLE)
with_role = call_llm(PROMPT_WITH_ROLE)

print(f"Email     : {email['subject']}")
print(f"Expected  : {email['correct']}")
print(f"No Role   : {no_role}")
print(f"With Role : {with_role}")

In [ ]:
email = {
    "subject": "SFDC integration not syncing",
    "sender" : "cto@bigclient.com",
    "body"   : "Salesforce stopped syncing leads 3 days ago. "
               "Sales team is blind. Losing deals.",
}

subject    = email['subject']
sender     = email['sender']
body       = email['body']
customer_mrr = 1500   # $1500/month — high value customer

PROMPT_CLASSIFIER = f"""You are an expert support email classifier.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Return ONLY: Category | Confidence (1-10)"""

PROMPT_PRIORITY = f"""You are a Customer Success Manager
with 5 years experience at a B2B SaaS company.

You understand:
- Customer MRR > $1000/month = high value = bump priority
- Sales team blocked = revenue impact = urgent
- SLAs: Enterprise (>$1000 MRR) = 2hr, Standard = 24hr

Given this classified ticket:
Category     : Technical
Subject      : {subject}
Body         : {body}
Customer MRR : ${customer_mrr}/month

Score priority 1-10:
10 = drop everything now
1  = can wait days

Return ONLY:
Priority : X/10
SLA      : X hours
Reason   : one line"""

category = call_llm(PROMPT_CLASSIFIER)
priority = call_llm(PROMPT_PRIORITY)

print(f"Step 1 — Category : {category}")
print(f"Step 2 — Priority :\n{priority}")

In [13]:
## session 4

email = {
    "subject": "Can't login — paid for annual plan last week",
    "body"   : "Our entire team can't login. I made the payment"
               "plan last week. Board demo in 3 hours.",
    "correct": "Billing"
}

subject = email['subject']
body    = email['body']

PROMPT_CURRENT = f"""You are an expert support email classifier.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Return ONLY: Category | Confidence (1-10)"""

result = call_llm(PROMPT_CURRENT)
print(f"Current Classifier : {result}")
print(f"Expected           : Billing")

Current Classifier : Category | Confidence (1-10)  
Technical | 9
Expected           : Billing


In [14]:
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- If root cause is payment → ALWAYS Billing
- Surface complaint does not decide the category
- ROOT CAUSE decides the category

Then classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""

result = call_llm(PROMPT_COT_V2)
print(result)

THINKING:
1. Surface complaint: Can't login.
2. Root cause: Payment issue related to account activation after the annual plan payment.
3. Team that OWNS this: Billing.
4. Business impact: The entire team is unable to access the service, which could jeopardize an important board demo scheduled in 3 hours.

CATEGORY: Billing


In [16]:

warmup_prompt = """You are a fintech support analyst. 
    
Before I give you a real email to classify, I want you to demonstrate 
your reasoning ability. Generate 3 different example emails a fintech 
support team might receive, and show step-by-step how you would 
classify each one. 

Cover different categories: payment failures, KYC issues, and fraud alerts.

Format each as:
EXAMPLE [N]:
Email: [email text you invent]
Reasoning: [step-by-step thinking]
Classification: [category + urgency]"""


result = call_llm(warmup_prompt)


In [18]:
print(result)

EXAMPLE 1:
Email: "Hello, I attempted to make a payment of $150 for my subscription, but I received an error message stating that the transaction failed. Can you please help me resolve this issue? Thank you!"
Reasoning: 
1. The email mentions a specific issue related to a payment attempt.
2. The user describes an error message received during the transaction process.
3. The request for assistance indicates that the user is seeking a resolution to a problem that is preventing them from completing a payment.
4. The urgency is moderate, as the user is likely looking to resolve the issue quickly to access their subscription.
Classification: Payment Failure + Moderate Urgency

---

EXAMPLE 2:
Email: "Dear Support Team, I recently submitted my KYC documents, but I received a notification that my account is still not verified. Can you please provide an update on the status of my verification? I need access to my funds urgently."
Reasoning:
1. The email focuses on the KYC (Know Your Customer) 

In [19]:
classify_prompt = f"""You are a fintech support analyst.

You just demonstrated your reasoning ability with these examples:
{result}

Now apply the SAME reasoning pattern to this real email:

Subject : {subject}
Body    : {body}

Follow the same format: Reasoning → Classification JSON."""

output = call_llm(classify_prompt)

In [20]:
print(output)

Reasoning:
1. The email is a follow-up regarding multiple issues related to the user's account, indicating ongoing problems that have not been resolved.
2. The user outlines several specific issues: a failed payment that still resulted in a charge, intermittent API access, SDK errors, and a pending integration request.
3. The mention of a contract renewal next month suggests a time-sensitive situation, as the user is evaluating whether to continue their relationship with the service.
4. The request to escalate the matter indicates a high level of urgency, as the user is seeking immediate attention to multiple critical issues that could impact their business operations.

Classification: Account Issues + High Urgency

```json
{
  "Classification": "Account Issues",
  "Urgency": "High"
}
```


In [ ]:
## self consitency

In [21]:
from collections import Counter

email = {
    "subject": "Re: Re: Fwd: Account Issue - URGENT",
    "body"   : "Hi, following up on my earlier emails. "
               "To summarize the situation: "
               "1) Last month our payment failed but we were still charged. "
               "2) Since then our API access has been intermittent. "
               "3) Our dev team says the SDK throws 403 errors randomly. "
               "4) We tried resetting API keys but same issue. "
               "5) Meanwhile we requested a Zapier integration 2 weeks ago. "
               "6) Our contract renews next month and we are evaluating "
               "whether to continue. "
               "Please escalate this to the right team immediately.",
    "correct": "Billing"
}

# ← ye update karo!
subject = email['subject']
body    = email['body']

PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Important:
- Root cause decides category, not surface words
- Pricing complaint → always Billing
- Feature requests alongside pricing → Billing wins
- 403 errors after payment failure → Billing not Technical

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""

print("Running 5 times with high temperature...")
print()

results = []

for i in range(5):
    result = call_llm(PROMPT_COT_V2)
    for line in result.split("\n"):
        if "CATEGORY" in line:
            category = line.replace("CATEGORY:", "").strip()
            results.append(category)
            print(f"Run {i+1} : {category}")

# vote count
votes  = Counter(results)
winner = votes.most_common(1)[0]

print()
print("=" * 40)
print("VOTE COUNT:")
for category, count in votes.items():
    bar = "█" * count
    print(f"  {category:20} : {bar} {count}/5")
print()
print(f"Winner     : {winner[0]}")
print(f"Confidence : {winner[1]/5*100:.0f}%")
print("=" * 40)

Running 5 times with high temperature...

Run 1 : Billing
Run 2 : Billing
Run 3 : Billing
Run 4 : Billing
Run 5 : Billing

VOTE COUNT:
  Billing              : █████ 5/5

Winner     : Billing
Confidence : 100%


In [ ]:
email = {
    "subject": "Evaluating if we stay on your platform",
    "body"   : "We've been using your product for 2 years. "
               "Lately evaluating alternatives. "
               "Main blockers: No Zapier, no bulk export, "
               "pricing jumped 40% at last renewal. "
               "Can we discuss?",
}

subject = email['subject']
body    = email['body']

PROMPT_TOT = f"""You are a Senior Customer Success Manager.

A customer sent this email:
Subject : {subject}
Body    : {body}

Explore 3 different response strategies.
For each — write the response AND score it.

STRATEGY 1: Immediate Discount Offer
Response: [write it]
Score: X/10
Why: [one line]

STRATEGY 2: Empathetic + Schedule a Call
Response: [write it]
Score: X/10
Why: [one line]

STRATEGY 3: Escalate to Account Manager
Response: [write it]
Score: X/10
Why: [one line]

BEST STRATEGY: [which one and why]

FINAL RESPONSE:
[write the winning response]"""

result = call_llm(PROMPT_TOT)
print(result)